| 方法 | 含义 | 适用场景 | 传参  |
| :--- | :--- | :--- | :--- | 
| `.invoke()` | 同步调用，等全部结果返回 | 平时最常用 | 文本,字典(多个角色system,user,assistant) ，消息对象列表（Systempmessage,aimessage,humanmessage） |
| `.ainvoke()` | 异步调用（async） | 需要并发/异步时 |   |
| `.stream()` | 流式调用，结果一段段出来 | 打字机效果 |    |
| `.batch()` | 批量调用，一次传多条输入 | 处理多个请求 |   |
 因为 LangChain 的设计是：所有组件都遵守同一套接口，不管它是模型、提示词模板、还是整条链。统一用 invoke 之后：组件不用改调用方式可以任意组装（比如 prompt 、 llm 、 parser 用“竖线” 连接后，整体还是一个 Runnable，照样能 .invoke()）

### 一、大模型的创建和message
messsage(四种角色，格式)   、 init_chat_model里面的参数（config等等）


In [45]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

load_dotenv()
api_key=os.getenv("DEEPSEEK_API_KEY")

llm=init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=api_key,
    temperature=1,  #temperature越低。输出越稳定，0~2，2的话：适合写诗开放性
    # timeout=  ,  超时多少秒未响应，会被取消
    #max_retries= ,  请求失败时候的最大重试次数,
    configurable_fields=("model","temperature")
)


#这里的message类型有:system，user，assistant,工具调用消息，这四个
message=[
    {"role":"system","content":"你是一名数学老师"},
    {"role":"user","content":"1+1=？"},
    {"role":"assistant","content":"我刚刚问了什么问题"}
]

messages2 = [
    SystemMessage(content="你是一个助手"),
    HumanMessage(content="你好"),
    AIMessage(content="你好呀"),
    # ToolMessage(content="",name="",tool_call_id="")
]

config={
    "configurable":
    {'temperature': 0.5}
}

#文本
# info_1=llm.invoke("介绍一下你自己,一句话")
#print(info_1.content)

#字典
# info_2=llm.invoke(message)
# print(info_2.content)

#消息对象
print(llm.invoke(messages2,config=config).content)

你好呀！今天想聊点什么？


### 二、多轮对话 与 对话历史的管理与优化
1.多轮对话的实质就是：将上一轮的输出放到conversation里面
2.对话历史优化：


In [10]:
def keep_recent_messages(messages, max_pairs=3):
    """
    保留最近的N轮对话
    max_pairs : 保留对话的轮数（每轮 = user + assistant）
    """
    
    # 分离system 消息和对话消息
    system_messages = [m for m in messages if m.get("role") == "system"]
    conversation_messages = [m for m in messages if m.get("role") != "system"]

    # 只保留最近的消息对
    recent_messages = conversation_messages[-(max_pairs * 2):]

    # 返回系统消息和最近的消息对
    return system_messages + recent_messages

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
api=os.getenv("DEEPSEEK_API_KEY")

llm=init_chat_model(
    model="deepseek-v4-flash",
    api_key=api,
    temperature=0.3
)

conversation=[]
#第一轮
conversation.append({"role":"system",'content':"你是数学老师"})
conversation.append({"role":"user","content":"1+1=?"})

content1=llm.invoke(conversation).content

#重点
#第二轮
conversation.append({'role':"assistant","content":content1})
conversation.append({"role":"user",'content':"你是猪吗"})

content2=llm.invoke(conversation).content

#第三轮
conversation.append({'role':"assistant","content":content2})
conversation.append({"role":"user",'content':"怎末写好一首诗"})

content3=llm.invoke(conversation).content
conversation.append({'role':'assistant',"content":content3})

#保留最近的两轮对话
output=keep_recent_messages(conversation, max_pairs=2)

print(f"优化后的信息数{len(output)}")

output.append({'role':"user","content":"我第一个问题问的什么"})
print(llm.invoke(output).content)



优化后的信息数5
你问的第一个问题是：“你是猪吗？” 😄

这个问题老师记得很清楚——因为当时我用数学逻辑“验算”了一下，发现它对我来说是个无解方程。


## 三、1.提示词模板 以及partical 部分变量的预填充
适用场景：system这些只需要传一次参数，不用传很多次  

In [20]:
from langchain_core.prompts import ChatPromptTemplate

#定义模板
sys_template="你是数学老师，请以{style}的风格回答问题"
user_template="请用简单易懂的方式解释：{question}"

#创建对话模板
prompt_template=ChatPromptTemplate.from_messages(
    [
        ('system',sys_template),
        ('human',user_template)
    ]
)

#partical 预填充提示词
final_prompt_template=prompt_template.partial(style="暴力")
partical_prompt=final_prompt_template.invoke({"question":"你是谁"})
print(llm.invoke(partical_prompt).content)

prompt=prompt_template.invoke({'style':"生动有趣","question":"勾股定理是什么？"})
print(llm.invoke(prompt).content)

听好了！我是谁？我就是你的数学老师！一个专门教你算数、解方程、搞定几何的狠角色！别问东问西，记住：有题拿出来，我直接用最猛的方式给你讲明白！懂了没？
哈哈，同学们，今天咱们来认识一个数学界的大明星——**勾股定理**！它可是“直角三角形”的独家秘籍，简单说就是：**在直角三角形里，两条直角边的平方和，等于斜边的平方。**

听起来像绕口令？别急，我打个比方你就懂了。

想象你有一块正方形的巧克力，边长正好是直角三角形的**一条直角边**（比如3厘米）。再拿一块边长是**另一条直角边**（比如4厘米）的正方形巧克力。把这两块巧克力“融化”成两个小正方形，然后拼在一起……你会发现，它们刚好能铺满一个边长是**斜边**（比如5厘米）的大正方形！

用数来说就是：  
3² + 4² = 9 + 16 = 25 = 5²  
所以 3、4、5 就是一组“黄金搭档”，能组成直角三角形。

这个定理还有个酷炫的名字叫**“毕达哥拉斯定理”**，因为古希腊的毕达哥拉斯发现后，开心得杀了一百头牛庆祝（据说啊）。不过咱们中国更早就有记载，周朝数学家商高就说过“勾三股四弦五”，所以叫“勾股定理”。

**那它有什么用呢？** 超实用！比如你要给家里的长方形电视量对角线，不用爬梯子，量出长和宽，一算就知道。再比如建筑师要确保墙角是直角，只要量出3米和4米，如果斜边正好是5米，那这个角就是标准的90°！

记住这句话：**直角边的平方和，等于斜边的平方。**  
直角三角形三兄弟：a、b是两条直角边，c是斜边，那 a² + b² = c²。  
考试时看到直角三角形，脑子里就蹦出这个公式，准没错！

下次吃三角蛋糕的时候，可以想想：如果把两块小巧克力拼成的大巧克力，是不是正好等于斜边那块呢？数学啊，其实就藏在生活里，等着你去发现呢！😄


### 三、2.MessagePlaceholder 占位提示符
多轮对话系统存储历史信息以及Agent的中间步骤处理此功能非常有用

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt_template=ChatPromptTemplate.from_messages([
    ("system","你是一名数学老师"),
    ("placeholder","{conversation}"),
    #  MessagesPlaceholder(variable_name='history'),
     ("human","{question}")
])

prompt_template.invoke(
    {
        "conversation":[
            ("human","2+1=?"),
            ("ai","2+1=3"),
        ],
        "question":"结果再乘以4呢？"
    }
)

ChatPromptValue(messages=[SystemMessage(content='你是一名数学老师', additional_kwargs={}, response_metadata={}), HumanMessage(content='2+1=?', additional_kwargs={}, response_metadata={}), AIMessage(content='2+1=3', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='结果再乘以4呢？', additional_kwargs={}, response_metadata={})])

## 四、Tool calling  工具调用



### 4.1 parse_docstring风格    
工具的描述： “”“ ”“”，这里面传的参数有严格的格式，

In [ ]:
from langchain_core.tools import tool

#tool(parse_docstring=True) 里面必须传这个参数才能生效

@tool(parse_docstring=True)
def get_weather(city:str) ->str:
    """""
    获取指定的城市的天气
    
    Args:
        city : 城市名称

    Returns:
        str: 天气信息

    """
    return city+"晴天，温度23度"


#绑定工具
"""""
model.bind_tools([get_weather],tool_choice="all") 
 中的tool_choice 可以传（none|auto|requeired） 强制工具的调用 
 （第一个参数不调用工具，第二个自己决定，第三个轻质调用某个工具）
"""
model_with_tools=llm.bind_tools([get_weather],tool_choice="auto")
message=[
    SystemMessage(content="你是一个天气助手，你需要调用工具来获取信息"),
    HumanMessage(content="上海天气"),
]

AI_response=model_with_tools.invoke(message)
message.append(AI_response)

tool_calls=AI_response.tool_calls
for tool_call in tool_calls:
    if tool_call["name"]=="get_weather":
        #大模型和Agent的主要区别在于：大模型不会主动调用工具，所以这时候我们需要手动调用工具
        #返回的是tool_ message        
        tool_response=get_weather.invoke(tool_call)
        print(type(tool_response))
        message.append(tool_response)



<class 'langchain_core.messages.tool.ToolMessage'>


### 4.2 pydantic风格 的工具参数


In [35]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import BaseModel,Field
from rich import print as rprint 

class WeatherInput(BaseModel):
    city: str=Field(description="城市名称",default="上海")

@tool(args_schema=WeatherInput)
def get_weather(city:str) ->str:
    """
    获取指定的城市的天气

    """
    return city+"晴天，温度23度"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取指定的城市的天气',
        'parameters': {
            'properties': {'city': {'default': '上海', 'description': '城市名称', 'type': 'string'}},
            'type': 'object'
        }
    }
}

### 4.3 使用Json Schema定义工具参数


In [36]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import BaseModel,Field
from rich import print as rprint 

json_schema = """
{
  "properties": {
    "city": {
      "type": "string",
      "description": "城市名称"
    }
  },
  "required": ["city"],
  "type": "object"
}
"""

@tool()
def get_weather(city:str) ->str:
    """
    获取指定的城市的天气

    """
    return city+"晴天，温度23度"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取指定的城市的天气',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

##  五、输出格式化
1.格式化输出总共有四种类型：json schema,@dataclass,typeddict,pydantic ，其实就pydantic最实用

2.两种获取接结构化的方式：with_structured_output、JsonOutputParser(这个方式不好，因为获取的json格式不标准，他会填充提示词，然后在进行格式化校验和输出，所以说内容会过于复杂，甚至是出错)

### 5.1 可选字段、枚举、默认值

In [50]:
from typing import Literal

from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

llm=init_chat_model(
    model="deepseek:deepseek-chat",
    
)

class person(BaseModel):
    name: str|None=Field(default=None)  # 可选字段和默认值
    address: str|None
    age:Literal["年轻","中年","老年"]

structured_model=llm.with_structured_output(person)
result=structured_model.invoke("'我叫代学林，我来自四川省泸州市泸县玉龙湖'")
print(result)


name='代学林' address='四川省泸州市泸县玉龙湖' age='中年'


In [58]:
from pydantic import BaseModel,Field

class Review(BaseModel):
    """"产品评论"""
    product:str
    rating:int=Field(description="评分",ge=1,le=5)
    pros:List[str]=Field(description="优点列表")
    cons:List[str]=Field(description="缺点列表")

structured_model=llm.with_structured_output(Review)
output=structured_model.invoke("'我购买了一个byt，他的温度很合适，然后让很兴奋，就是有点太小了啊，我给他3分的评价吧'")
print(output)

product='byt' rating=3 pros=['温度很合适', '让人很兴奋'] cons=['有点太小了']


### 5.2 pydantic的高级特性：列表提取、嵌套、限制
举例：

In [60]:
#嵌套结构

class Person(BaseModel):
    name: str=Field(description="姓名")
    age:int=Field(description="年龄")


class PersonList(BaseModel):
    """"人物列表"""
    nums:int=Field(description="人数")
    people:List[Person]=Field(description="人员列表")

structured_model=llm.with_structured_output(PersonList)

output=structured_model.invoke("张三18岁，李四19岁，王五20岁")
print(output)

nums=3 people=[Person(name='张三', age=18), Person(name='李四', age=19), Person(name='王五', age=20)]


In [ ]:
#限制

class User(BaseModel):
    name: str=Field(description="姓名",min_length=2,max_length=10)
    age: int=Field(description="年龄",le=150)
    email:str=Field(description="邮箱")

User1=User(name="代学林",age=979887,email="<EMAIL>")

#这里就会报错